# Callbacks

Er zijn heel wat extra functionaliteiten die je kan toevoegen aan het trainingsproces.
De meestgebruikte zijn: 
* EarlyStopping
* ModelCheckpoint
* Tensorboard
* LearningRate schedulers

Ook kan je custom callback functions maken om je eigen functionaliteiten toe te voegen.

In de klassieke pytorch manier schrijf je je eigen trainings-lus en is het dus eenvoudig om alle gewenste code toe te voegen. 
De efficientere manier om modellen te bouwen via Keras abstraheerd echter de trainingslus en train je door middel van de fit()-methode.
Deze manier maakt het echter onmogelijk om extra functionaliteiten toe te voegen. 

Hieronder vind je een voorbeeld van hoe je callbacks kan toevoegen aan een model gebouwd met keras op een pytorch backend

In [ ]:
import numpy as np
from keras_core.models import Sequential
from keras_core.layers import Dense
from keras_core.callbacks import EarlyStopping

# Maak een eenvoudig neuraal netwerk
model = Sequential([
    Dense(10, activation='relu', input_shape=(10,)),
    Dense(1)
])

# Compile het model
model.compile(optimizer='adam', loss='mean_squared_error')

# Genereer dummy data
x_train = np.random.rand(100, 10)
y_train = np.random.rand(100, 1)

# Maak een EarlyStopping callback
early_stop = EarlyStopping(monitor='val_loss', patience=2)

# Train het model met de callback
model.fit(x_train, y_train, epochs=100, validation_split=0.2, callbacks=[early_stop])


# Dashboard for opvolgen experimenten, evaluaties, ....

Hieronder staat een voorbeelden van hoe een dashboard kan gebruikt worden met pytorch.
Ik heb hieronder gekozen voor wandb (weights and biases) maar alternatieven zijn tensorboard, mlflow, visdom, ...

In [ ]:
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
import torch.optim as optim

# Dummy data
X = torch.randn(100, 10)
y = torch.randn(100, 1)

# Model
model = nn.Sequential(
    nn.Linear(10, 5),
    nn.ReLU(),
    nn.Linear(5, 1)
)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# Start MLflow run
with mlflow.start_run(run_name="pytorch_example"):
    for epoch in range(10):
        optimizer.zero_grad()
        outputs = model(X)
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

        # log loss
        mlflow.log_metric("loss", loss.item(), step=epoch)

    # log model
    mlflow.pytorch.log_model(model, "model")


Door bovenstaande code uit te voeren wordt een map aangemaakt waar de nodige logs in bewaard worden.
Deze kunnen daarna gevisualiseerd worden met behulp van het volgende terminal-commando.
In de output van het commando krijg je een url te zien met de link waar je de tensorboard applicatie kan bekijken.
Met de standaardconfiguratie van de docker container moet je de applicatie kunnen bereiken via de link [localhost:6006](http://localhost:6006)

In [ ]:
!mlflow ui --host 0.0.0.0 --port 5000